# ORA
Over-Representation Analysis

I've identified some groups of DEGs that are of interest from [upset plots](https://github.com/jgmcdonough/CE24_RNA-seq/blob/main/analysis/diff_expression/phase2_v_phase2/new_refGenome/upset_p2.v.p2.ipynb) - I want to now know the functions of these groups

I can't do GSEA with this data because GSEA is rank based - instead doing an ORA because I just supply it with a gene list of interest and compare against the universal list

## 0. load libraries

In [2]:
library(tidyverse)
library(clusterProfiler) # for GSEA()
library(enrichplot) # for enrichment visuals 
library(GO.db) # for gene ontology database
library(UpSetR) # for Cvirginica annotations
library(patchwork) # for arranging multiple plots

## 1. read CSVs
For this analysis, I need a **universal list** (all genes tested) and a **gene list** that contains genes of interest (in this case, DEGs)

#### universal list

In [5]:
universal <- read.csv('/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/new_refGenome/deseq_res/bc_hc.csv')$Gene

length(universal)
head(universal)

[1] 29019

[1] "LOC144621260" "LOC144621269" "LOC111120925" "Trnae-cuc-2"  "Trnae-cuc-3" 
[6] "LOC144621283"

#### DEGs of interest list

In [9]:
earlyLife_shared <- read.csv('/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/new_refGenome/deg_interest/earlyLife_sharedAll.csv')$gene
earlyLife_shared

warm_shared <- read.csv('/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/new_refGenome/deg_interest/warm_sharedAll.csv')$gene
warm_shared

[1] "LOC111102028" "LOC111127247" "LOC144622232" "LOC144624450" "LOC144623104"
 [6] "LOC111116266" "LOC144624833" "LOC144624971" "LOC144625005" "LOC111116296"
[11] "LOC144626817" "LOC111099077" "LOC111136642" "LOC111132872" "LOC144627224"
[16] "LOC144619016" "LOC144619236" "LOC111128708" "LOC144619617" "LOC111130993"
[21] "LOC111118775" "LOC111119065" "LOC111118161" "LOC111114163" "LOC144621228"
[26] "LOC144621324" "LOC111114643" "LOC144621941"

[1] "LOC111102028" "LOC111127247" "LOC144624450" "LOC144623813" "LOC144623104"
 [6] "LOC111113307" "LOC111116266" "LOC144624833" "LOC144624971" "LOC144626817"
[11] "LOC111099077" "LOC111136642" "LOC111110618" "LOC111108826" "LOC144618695"
[16] "LOC144619223" "LOC144619016" "LOC111137950" "LOC144619031" "LOC111119065"
[21] "LOC111114254" "LOC111114163" "LOC144621228" "LOC144621324" "LOC144621941"

## 2. create term2gene and term2name

In [11]:
# col1 = gene ID
# col2 = GO ID 
gene2go <- read.csv('/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/annotations/newRef_geneGO.csv')
head(gene2go)

,gene,Gene.Ontology.IDs
,<chr>,<chr>
1,LOC111099029,GO:0005261;GO:0005886;GO:0030001;GO:0098655
2,LOC111099032,GO:0004930;GO:0005886;GO:0007186
3,LOC111099037,GO:0000978;GO:0000981;GO:0006355
4,LOC111099039,GO:0004930;GO:0005886;GO:0007189
5,LOC111099040,GO:0005515;GO:0007169;GO:0035556
6,LOC111099041,GO:0016020;GO:0022857;GO:0055085


re-format for correct input for `GSEA()` - two columns, one for GO term and one for gene ID

In [12]:
term2gene <- gene2go %>%
  mutate(GO_terms = strsplit(Gene.Ontology.IDs, ",\\s*|;\\s*|`")) %>%  # Split by comma, semicolon, or backtick
  unnest(GO_terms) %>%
  filter(grepl("^GO:", GO_terms)) %>%  # Keep only valid GO terms
  dplyr::select(term = GO_terms, gene = gene)

class(term2gene)
str(term2gene)
head(term2gene)

[1] "tbl_df"     "tbl"        "data.frame"

tibble [39,588 × 2] (S3: tbl_df/tbl/data.frame)
 $ term: chr [1:39588] "GO:0005261" "GO:0005886" "GO:0030001" "GO:0098655" ...
 $ gene: chr [1:39588] "LOC111099029" "LOC111099029" "LOC111099029" "LOC111099029" ...


term,gene
<chr>,<chr>
GO:0005261,LOC111099029
GO:0005886,LOC111099029
GO:0030001,LOC111099029
GO:0098655,LOC111099029
GO:0004930,LOC111099032
GO:0005886,LOC111099032


get term names for GO IDs

In [13]:
# Extract GO term descriptions
go_terms <- unique(term2gene$term)

# Get descriptions from GO.db
term2name <- data.frame(
  term = go_terms,
  name = sapply(go_terms, function(x) {
    tryCatch({
      Term(GOTERM[[x]])
    }, error = function(e) {
      NA_character_
    })
  })
)

# Remove NAs
term2name <- term2name[!is.na(term2name$name), ]

# View
head(term2name)    

,term,name
,<chr>,<chr>
GO:0005261,GO:0005261,monoatomic cation channel activity
GO:0005886,GO:0005886,plasma membrane
GO:0030001,GO:0030001,metal ion transport
GO:0098655,GO:0098655,monoatomic cation transmembrane transport
GO:0004930,GO:0004930,G protein-coupled receptor activity
GO:0007186,GO:0007186,G protein-coupled receptor signaling pathway


## 3. `enricher`
universal enrichment analyzer from clusterProfiler - hypergeometric test

provide a pre-defined list of interesting genes (DEGs) and comparing against a universal set

### early life stress
#### general response - shared DEGs
DEGs that were shared among HC, WC, and BC comparisons against CC oysters

In [14]:
# CC vs. WC
res_earlyShared <- enricher(
    gene = earlyLife_shared,
    TERM2GENE = term2gene,
    TERM2NAME = term2name,
    pvalueCutoff = 0.05,
    pAdjustMethod = "BH",
    universe = universal)

res_earlyShared <- as.data.frame(res_earlyShared)
head(res_earlyShared)

ID,Description,GeneRatio,BgRatio,pvalue,p.adjust,qvalue,geneID,Count
<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<int>


no over-represented gene ontologies :(

### warming across life stages
#### general response - shared DEGs
DEGs that were shared among WW, WC, and CW comparisons against CC oysters

In [15]:
# CC vs. WC
res_warmShared <- enricher(
    gene = warm_shared,
    TERM2GENE = term2gene,
    TERM2NAME = term2name,
    pvalueCutoff = 0.05,
    pAdjustMethod = "BH",
    universe = universal)

res_warmShared <- as.data.frame(res_warmShared)
head(res_warmShared)

ID,Description,GeneRatio,BgRatio,pvalue,p.adjust,qvalue,geneID,Count
<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<int>


no over-represented gene ontologies :(

which i guess makes sense because most of these genes are uncharacterized